# Remove comments and neutralize public repo code
This notebook strips comments from the repository files, normalizes formatting, and preserves functionality before public release.

In [ ]:
import re
from pathlib import Path

# Define the target repository files for cleanup
TARGET_FILES = [
    'config.js',
    'data.js',
    'index.html',
    'script.js',
    'styles.css',
]

# Helper to load file contents and preserve backups
backups = {}
for fname in TARGET_FILES:
    path = Path(fname)
    if path.exists():
        backups[fname] = path.read_text(encoding='utf-8')


In [ ]:
def strip_comments(text: str, ext: str) -> str:
    if ext == '.html':
        # Remove HTML comments
        return re.sub(r'<!--.*?-->', '', text, flags=re.S)
    if ext == '.css':
        return re.sub(r'/\*.*?\*/', lambda m: '\n' * m.group(0).count('\n'), text, flags=re.S)
    if ext in ('.js', '.jsx', '.ts', '.tsx'):
        result = []
        i = 0
        n = len(text)
        in_str = None
        escaped = False
        while i < n:
            c = text[i]
            nxt = text[i + 1] if i + 1 < n else ''
            if in_str:
                result.append(c)
                if escaped:
                    escaped = False
                elif c == '\\':
                    escaped = True
                elif c == in_str:
                    in_str = None
                i += 1
                continue
            if c in ('"', "'", '`'):
                in_str = c
                result.append(c)
                i += 1
                continue
            if c == '/' and nxt == '/':
                i += 2
                while i < n and text[i] != '\n':
                    i += 1
                continue
            if c == '/' and nxt == '*':
                i += 2
                while i < n and not (text[i] == '*' and i + 1 < n and text[i + 1] == '/'):
                    if text[i] == '\n':
                        result.append('\n')
                    i += 1
                i += 2
                continue
            result.append(c)
            i += 1
        return ''.join(result)
    return text

for fname, original in backups.items():
    ext = Path(fname).suffix.lower()
    cleaned = strip_comments(original, ext)
    if cleaned != original:
        Path(fname).write_text(cleaned, encoding='utf-8')
        print(f'Updated {fname}')
    else:
        print(f'No comments removed from {fname}')
